In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import nltk
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

load_dotenv()

In [ ]:
# download if you haven't before
# nltk.download('wordnet')  

In [ ]:
from nltk.corpus import wordnet as wn

def build_embedding_text(synset):
    definition = synset.definition()
    lemmas = ", ".join([str(lemma.name().replace("_", " ")) for lemma in synset.lemmas()])
    embedding = f"""The definition of {synset.name().split('.', 1)[0]} is {definition}. The words in this synset are: {lemmas}."""

    
    if len(synset.hypernyms()) != 0:
        hypernyms = ", ".join([str(hypernym.name()) for hypernym in synset.hypernyms()])
        embedding += f""" Hypernyms of this synset are: {hypernyms}."""
    if len(synset.examples()) != 0:
        examples = ", ".join([str(f"\"{example}\"") for example in synset.examples()])
        embedding += f""" Examples are: {examples}."""
    return embedding

all_embedding_sentences = []
metadata = []
for synset in wn.all_synsets():
    # if len(synset.lemmas()) >= 2:   # add if database building is slow since a lot of synsets seem to have only one word in them
    all_embedding_sentences.append(build_embedding_text(synset))
    metadata.append({
        "synset": synset.name(),
        "definition": synset.definition(),
        "lemmas": [lemma.name() for lemma in synset.lemmas()],
        "hypernyms":[hypernym.name() for hypernym in synset.hypernyms()],
        "pos": synset.pos()
    })

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(all_embedding_sentences, show_progress_bar=True)

In [ ]:
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms

normalized_embeddings = normalize(embeddings)
num_dimensions = embeddings.shape[1]
index = faiss.IndexFlatIP(num_dimensions)
index.add(normalized_embeddings) 

faiss.write_index(index, "wordnet.index")

with open("metadata.pkl", "wb") as f:
    pickle.dump(({"documents": all_embedding_sentences, "metadata": metadata}), f)


# Start here after running above once

In [32]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
import json


load_dotenv()

True

In [2]:
# Load FAISS index
index = faiss.read_index("wordnet.index")

# Load documents + metadata
with open("metadata.pkl", "rb") as f:
    data = pickle.load(f)

documents = data["documents"]
metadata = data["metadata"]

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms

def search(query, k=2):
    
    query_vec = normalize(model.encode([query]))
    distances, indices = index.search(query_vec, k)

    results = []
    for i in indices[0]:
        results.append(documents[i])

    return results

In [29]:
search("dog, bird")

['The definition of bird_dog is a gun dog trained to locate or retrieve birds. The words in this synset are: bird dog. Hypernyms of this synset are: sporting_dog.n.01.',
 'The definition of bird is the flesh of a bird or fowl (wild or domestic) used as food. The words in this synset are: bird, fowl. Hypernyms of this synset are: meat.n.01.']

In [ ]:
# Install if needed
# %pip install ipywidgets

import ipywidgets as widgets
from IPython.display import display, Markdown

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

messages = [
    {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. You have access to WordNet to analyze semantic relationships between words.
        Use the analyze_wordnet_relationships function to:
        
Find similarities between words
Get hypernyms (broader categories) for words
Get definitions of words
  The function returns all supported analyses at once.

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}

        Think step by step and use WordNet data to support your reasoning.
        Return ONLY a JSON array with exactly 4 objects, each with:
        
"answerDescription": a short ALL-CAPS description of the category
"words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]
        """}
]

def ask_gpt(words, prompt):
    data=[]
    for word in words:
        data.append(str(search(word)))
    new_prompt = "\n".join(datum for datum in data).append(prompt)
    messages.append({"role": "user", "content": new_prompt})
    
    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=messages
    )
    
    reply = completion.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    
    return reply  

print(ask_gpt([
        "LASER",
        "PLUCK",
        "THREAD",
        "WAX",
        "COIL",
        "SPOOL",
        "WIND",
        "WRAP",
        "HONEYCOMB",
        "ORGANISM",
        "SOLAR PANEL",
        "SPREADSHEET",
        "BALL",
        "MOVIE",
        "SCHOOL",
        "VITAMIN"
    ]))
# group_prompt = (
#         "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
#         + "and return only the groups as lists: " )

TypeError: sequence item 0: expected str instance, list found

In [37]:
# Helper functions for wordnet, evaluating connections, and other related tasks

from nltk.corpus import wordnet as wn

def get_word_synsets(words):
    """Return a dict mapping each word to its unique lemma names from WordNet."""
    result = {}
    for word in words:
        synsets = wn.synsets(word.lower().replace(" ", "_"))
        result[word] = list(dict.fromkeys(
            l.name().replace("_", " ") for s in synsets for l in s.lemmas()
        ))
    return result

## Evaluation metrics for predicted groups vs actual answers

def game_completion_rate(all_predicted, all_actual):
    """All 4 groups exactly correct → puzzle is 'completed'."""
    if not all_predicted:
        return 0.0
    completed = 0
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        pred_sets   = [set(p["words"]) for p in predicted]
        if all(ps in actual_sets for ps in pred_sets):
            completed += 1
    return completed / len(all_predicted)


def overall_accuracy(all_predicted, all_actual):
    """Proportion of predicted groups that exactly match a ground-truth group."""
    total, correct = 0, 0
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        for pred in predicted:
            total += 1
            if set(pred["words"]) in actual_sets:
                correct += 1
    return correct / total if total else 0.0


def partial_accuracy_rate(all_predicted, all_actual):
    """
    Proportion of predicted groups whose best overlap with any ground-truth group
    is exactly 3 or exactly 2 words (i.e. near-correct but not exact).
    Returns dict: {3: rate_for_3_overlap, 2: rate_for_2_overlap}
    """
    total = 0
    counts = {2: 0, 3: 0}
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        for pred in predicted:
            pred_set = set(pred["words"])
            best_overlap = max(len(pred_set & a_set) for a_set in actual_sets)
            total += 1
            if best_overlap in counts:
                counts[best_overlap] += 1
    return {k: v / total if total else 0.0 for k, v in counts.items()}



## Evaluation helpers — run once, then call evaluate_run() in the cells below

from IPython.display import display, HTML

def _badge(text, bg, fg="#fff"):
    return (f'<span style="background:{bg};color:{fg};padding:2px 8px;'
            f'border-radius:4px;font-size:0.85em;font-weight:bold">{text}</span>')

def render_puzzle(idx, total, contest, raw_reply, predicted, actual, scores):
    correct_groups = scores["correct_groups"]
    completed      = scores["completed"]
    group_acc      = scores["group_acc"]
    actual_sets    = [set(a["words"]) for a in actual]

    status_badge = (_badge("✓ SOLVED", "#2e7d32") if completed
                    else _badge(f"✗ {correct_groups}/4", "#c62828"))

    html = []
    html.append('<div style="border:1px solid #555;border-radius:8px;padding:16px;'
                'margin:12px 0;font-family:monospace">')

    # Header
    html.append('<div style="display:flex;align-items:center;gap:12px;margin-bottom:12px">')
    html.append(f'<b style="font-size:1.1em">[{idx}/{total}] {contest}</b>')
    html.append(status_badge)
    html.append(f'<span style="color:#888;font-size:0.85em">'
                f'acc {group_acc:.0%} | partial-3 {scores["partial_3"]:.0%} | partial-2 {scores["partial_2"]:.0%}'
                f'</span>')
    html.append('</div>')

    # Raw reply (collapsible)
    escaped = raw_reply.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    html.append('<details style="margin-bottom:12px">')
    html.append('<summary style="cursor:pointer;color:#90caf9">▶ Raw model reply</summary>')
    html.append(f'<pre style="background:#1e1e1e;color:#d4d4d4;padding:10px;border-radius:6px;'
                f'overflow-x:auto;white-space:pre-wrap;font-size:0.85em;margin-top:6px">{escaped}</pre>')
    html.append('</details>')

    # Side-by-side table
    html.append('<table style="width:100%;border-collapse:collapse;font-size:0.9em">')
    html.append('<tr>'
                '<th style="text-align:left;padding:4px 8px;border-bottom:1px solid #444;width:50%;color:#aaa">Predicted</th>'
                '<th style="text-align:left;padding:4px 8px;border-bottom:1px solid #444;width:50%;color:#aaa">Actual</th>'
                '</tr>')

    for row in range(max(len(predicted), len(actual))):
        html.append('<tr style="vertical-align:top">')

        if row < len(predicted):
            pred = predicted[row]
            ps = set(pred["words"])
            is_correct = ps in actual_sets
            bg = "#1b3a1b" if is_correct else "#3a1a1a"
            icon_color = "#66bb6a" if is_correct else "#ef5350"
            icon = "✓" if is_correct else "✗"
            label = pred.get("answerDescription", "?")
            if is_correct:
                words_display = [f'<span style="color:#66bb6a">{w}</span>' for w in pred["words"]]
                note = ""
            else:
                best_a = max(actual, key=lambda a: len(ps & set(a["words"])))
                best_set = set(best_a["words"])
                words_display = [
                    f'<span style="color:{"#66bb6a" if w in best_set else "#ef5350"}">{w}</span>'
                    for w in pred["words"]
                ]
                overlap = len(ps & best_set)
                note = (f'<div style="color:#888;font-size:0.8em;margin-top:2px">'
                        f'best match: <i>{best_a["answerDescription"]}</i> ({overlap}/4)</div>')
            html.append(
                f'<td style="padding:5px 8px;background:{bg};border-radius:4px">'
                f'<span style="color:{icon_color}">{icon}</span> '
                f'<b style="color:#ccc">{label}</b><br>'
                f'{", ".join(words_display)}{note}</td>'
            )
        else:
            html.append('<td></td>')

        if row < len(actual):
            a = actual[row]
            diff_colors = {"Yellow": "#f9a825", "Green": "#2e7d32",
                           "Blue": "#1565c0", "Purple": "#6a1b9a"}
            cat_color = diff_colors.get(a.get("difficulty", ""), "#555")
            html.append(
                f'<td style="padding:5px 8px;background:#1e1e2e;border-radius:4px">'
                f'<b style="color:{cat_color}">{a["answerDescription"]}</b><br>'
                f'<span style="color:#ccc">{", ".join(a["words"])}</span></td>'
            )
        else:
            html.append('<td></td>')

        html.append('</tr>')

    html.append('</table></div>')
    return "".join(html)


def evaluate_run( raw_replies, all_actual):
    """Evaluate a list of raw model replies against ground-truth answers. Returns per_puzzle_scores."""
    display(HTML(
        f'<h2 style="font-family:monospace;border-bottom:2px solid #555;padding-bottom:6px">'
    ))

    predicted_all  = []
    per_puzzle_scores = []

    for i, (raw_reply, actual) in enumerate(zip(raw_replies, all_actual)):
        try:
            s = raw_reply.find('[')
            e = raw_reply.rfind(']') + 1
            predicted = json.loads(raw_reply[s:e])
        except Exception as ex:
            print(f"[{i+1}] Failed to parse JSON: {ex}")
            predicted = []
        predicted_all.append(predicted)

        actual_sets = [set(a["words"]) for a in actual]
        pred_sets   = [set(p["words"]) for p in predicted]

        correct_groups = sum(1 for ps in pred_sets if ps in actual_sets)
        completed      = int(all(ps in actual_sets for ps in pred_sets) and len(pred_sets) == 4)
        partial_3 = sum(
            1 for ps in pred_sets
            if max((len(ps & a) for a in actual_sets), default=0) >= 3
        ) / max(len(pred_sets), 1)
        partial_2 = sum(
            1 for ps in pred_sets
            if max((len(ps & a) for a in actual_sets), default=0) >= 2
        ) / max(len(pred_sets), 1)
        group_acc = correct_groups / max(len(pred_sets), 1)

        scores = dict(correct_groups=correct_groups, completed=completed,
                      group_acc=group_acc, partial_3=partial_3, partial_2=partial_2)
        per_puzzle_scores.append({"contest": pool[i]["contest"], **scores})
        display(HTML(render_puzzle(i + 1, len(pool), pool[i]["contest"],
                                   raw_reply, predicted, actual, scores)))

    # Summary table
    n = len(per_puzzle_scores)
    avg_gcr = sum(s["completed"]  for s in per_puzzle_scores) / n
    avg_acc = sum(s["group_acc"]  for s in per_puzzle_scores) / n
    avg_p3  = sum(s["partial_3"]  for s in per_puzzle_scores) / n
    avg_p2  = sum(s["partial_2"]  for s in per_puzzle_scores) / n

    header = (
        '<tr style="color:#aaa;border-bottom:1px solid #555">'
        '<th style="padding:4px 12px;text-align:left">Contest</th>'
        '<th style="padding:4px 12px">Solved</th>'
        '<th style="padding:4px 12px">Group Acc</th>'
        '<th style="padding:4px 12px">Partial-3</th>'
        '<th style="padding:4px 12px">Partial-2</th></tr>'
    )
    rows = "".join(
        f'<tr><td style="padding:4px 12px">{s["contest"]}</td>'
        f'<td style="padding:4px 12px;text-align:center">{"✓" if s["completed"] else "✗"}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["group_acc"]:.0%}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["partial_3"]:.0%}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["partial_2"]:.0%}</td></tr>'
        for s in per_puzzle_scores
    )
    avg_row = (
        f'<tr style="border-top:2px solid #666;font-weight:bold">'
        f'<td style="padding:6px 12px">Average ({n})</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_gcr:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_acc:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_p3:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_p2:.1%}</td></tr>'
    )
    display(HTML(
        f'<h3 style="font-family:monospace;margin-top:24px">Summary — {n} puzzle{"s" if n!=1 else ""}</h3>'
        f'<table style="border-collapse:collapse;font-family:monospace;font-size:0.9em">'
        f'{header}{rows}{avg_row}</table>'
    ))

    return per_puzzle_scores, dict(gcr=avg_gcr, acc=avg_acc, p3=avg_p3, p2=avg_p2)

In [35]:
_NYT_CONNECTION_FILE = "../data/ConnectionsFinalDataset.json"
#_NYT_CONNECTION_FILE = '/home/vgupta8/cs429/nyt-connections-rag/data/ConnectionsFinalDataset.json'
_DIFFICULTY = 4
_GAME_DATE = ""
_NUMBER_OF_CONNECTIONS = 4

In [36]:
import ipywidgets as widgets
from IPython.display import display, Markdown


import json

import os
from dotenv import load_dotenv
from openai import OpenAI
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

load_dotenv()



client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)



# Function definitions for GPT
tools = [
    {
        "type": "function",
        "function": {
            "name": "analyze_wordnet_relationships",
            "description": "Analyze semantic relationships between words using WordNet",
            "parameters": {
                "type": "object",
                "properties": {
                    "words": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of words to analyze"
                    }
                },
                "required": ["words"]
            }
        }
    }
]
def analyze_wordnet_relationships():
    return ""

def ask_gpt(messages, prompt, tools = None):
    # Use a fresh local conversation each call, keeping only the system prompt
    local_messages = [messages[0], {"role": "user", "content": prompt}]

    kwargs = {}
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=local_messages,
        **kwargs
    )

    if completion.choices and len(completion.choices) > 0:
        message = completion.choices[0].message
        tool_calls = getattr(message, "tool_calls", None)
        print("Tools:", getattr(message, "tool_calls", ""))
        if tool_calls:
            # Append the assistant's reply (with tool_calls) to the conversation
            local_messages.append(message)

            for tool_call in tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                if tool_name == "analyze_wordnet_relationships":
                    tool_result = analyze_wordnet_relationships(**tool_args)
                else:
                    tool_result = {"error": f"Unknown tool: {tool_name}"}

                # tool_call_id is required to match this result to the right call
                local_messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result)
                })

            followup = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=local_messages
            )
            if followup.choices:
                return followup.choices[0].message.content

        return getattr(message, "content", None)
    return str(completion)


#
puzzle = {'date': '2024/04/10',
 'contest': 'NYT Connections 304 - April 10th, 2024',
 'words': ['CHAIR',
  'CHIEF',
  'DIRECTOR',
  'HEAD',
  'FIELD',
  'GREEN',
  'GROUNDS',
  'LAWN',
  'CHEST',
  'COCO',
  'HAZEL',
  'PEA',
  'BROWN',
  'DOGS',
  'FICTION',
  'UNCHAINED'],
 'answers': [{'answerDescription': 'PERSON IN CHARGE',
   'words': ['CHAIR', 'CHIEF', 'DIRECTOR', 'HEAD']},
  {'answerDescription': 'GRASSY AREA',
   'words': ['FIELD', 'GREEN', 'GROUNDS', 'LAWN']},
  {'answerDescription': 'WORDS BEFORE “NUT”',
   'words': ['CHEST', 'COCO', 'HAZEL', 'PEA']},
  {'answerDescription': 'SECOND WORDS IN TARANTINO MOVIES',
   'words': ['BROWN', 'DOGS', 'FICTION', 'UNCHAINED']}],
 'difficulty': 4.0}

# function for Rag search
def RAG_search(words, k=5):
    """Return FAISS search results for each pair of words.""" 
    ##probably need a better explanation there, helppppp
    
    pair_results = {}
    for i, word1 in enumerate(words):
        for j, word2 in enumerate(words):
            if i < j:
                results = search(f"{word1} {word2}", k=k)
                pair_results[f"{word1}-{word2}"] = results
    return pair_results


import json
import random

with open(_NYT_CONNECTION_FILE) as f:
    dataset = json.load(f)

##Filters
DIFFICULTY            = _DIFFICULTY             # Set to None or "" to skip difficulty filter
GAME_DATE             = _GAME_DATE              # Set to None or "" to skip date filter (format: "YYYY/MM/DD")
NUMBER_OF_CONNECTIONS = _NUMBER_OF_CONNECTIONS  # Number of puzzles to randomly sample; set to None or 0 to use all

pool = dataset
if DIFFICULTY:
    pool = [p for p in pool if p["difficulty"] == DIFFICULTY]
if GAME_DATE:
    pool = [p for p in pool if p["date"] == GAME_DATE]
if NUMBER_OF_CONNECTIONS:
    pool = random.sample(pool, min(NUMBER_OF_CONNECTIONS, len(pool)))

if not pool:
    raise ValueError(f"No puzzles found for DIFFICULTY={DIFFICULTY!r}, GAME_DATE={GAME_DATE!r}")


print(f"Selected {len(pool)} puzzle(s) of difficulty {DIFFICULTY} and date '{GAME_DATE}' for evaluation.")


#----------------With RAG------------------------------
raw_replies_with_rag = []
all_actual_with_rag = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    Semantic_info = RAG_search(puzzle["words"], k=2)
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. You have access to WordNet to analyze semantic relationships between words.

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}

        Think step by step and use WordNet data to support your reasoning.
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]
        
        use below semantics infor to support the analysis:
        {json.dumps(Semantic_info, indent=2)}
        
        
        
        """}
        ]
    
    
    
    #------------------------------------------


    # Test GPT with RAG
    print("\nTesting GPT with RAG grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    
    raw_reply = ask_gpt(messages, group_prompt, tools=None)
    raw_replies_with_rag.append(raw_reply)
    all_actual_with_rag.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")

Selected 4 puzzle(s) of difficulty 4 and date '' for evaluation.

Testing GPT with RAG grouping...
Tools: None
  [1/4] done: NYT Connections 172 - November 30th, 2023

Testing GPT with RAG grouping...
Tools: None
  [2/4] done: NYT Connections 314 - April 20th, 2024

Testing GPT with RAG grouping...
Tools: None
  [3/4] done: NYT Connections 283 - March 20th, 2024

Testing GPT with RAG grouping...
Tools: None
  [4/4] done: NYT Connections 304 - April 10th, 2024


In [42]:
scores_rag, summary_rag = evaluate_run(
    raw_replies_with_rag,
    all_actual_with_rag,
)

Predicted,Actual
"✓ AVOIDANCE ACTIONSDODGE, DUCK, ESCAPE, SKIRT","AVOIDDODGE, DUCK, ESCAPE, SKIRT"
"✗ BIRDSGOOSE, BIRDS, ROBIN, HOBBESbest match: SIDEKICKS (3/4)","HITCHCOCK MOVIESBIRDS, NOTORIOUS, REBECCA, ROPE"
"✗ NOTORIOUS NAMESNOTORIOUS, REBECCA, WATSON, HOBBESbest match: HITCHCOCK MOVIES (2/4)","SIDEKICKSGOOSE, HOBBES, ROBIN, WATSON"
"✓ SAYING AND COMMUNICATIONSAY, STRING, CREAM, COTTAGE","___ CHEESECOTTAGE, CREAM, SAY, STRING"


Predicted,Actual
"✓ NONSENSE TERMSBUNK, CROCK, HOGWASH, HORSEFEATHERS","BALDERDASHBUNK, CROCK, HOGWASH, HORSEFEATHERS"
"✗ TOOLSHAMMER, PITCHFORK, BATON, HITCHbest match: TRACK AND FIELD EQUIPMENT (2/4)","TRACK AND FIELD EQUIPMENTBATON, HAMMER, HURDLE, POLE"
"✗ KNOTSBOWLINE, HITCH, SHEEPSHANK, HURDLEbest match: TYPES OF KNOTS (3/4)","PARTS OF A DEVIL COSTUMEGOATEE, HORNS, PITCHFORK, TAIL"
"✗ BODY PARTS OF ANIMALSGOATEE, HORNS, TAIL, BENDbest match: PARTS OF A DEVIL COSTUME (3/4)","TYPES OF KNOTSBEND, BOWLINE, HITCH, SHEEPSHANK"


Predicted,Actual
"✓ SPORTS VENUESASTROTURF, JUMBOTRON, SCOREBOARD, SKYBOX","SEEN AT A SPORTS STADIUMASTROTURF, JUMBOTRON, SCOREBOARD, SKYBOX"
"✓ CAMERA BRANDSFUJIFILM, HASSELBLAD, OLYMPUS, POLAROID","CAMERA BRANDSFUJIFILM, HASSELBLAD, OLYMPUS, POLAROID"
"✗ PASTA SAUCESBOLOGNESE, NEAPOLITAN, PARMESAN, CREAMSICLEbest match: ITALIAN DEMONYMS (3/4)","ITALIAN DEMONYMSBOLOGNESE, NEAPOLITAN, PARMESAN, VENETIAN"
"✗ MONUMENTSJOURNEYMAN, KISSCAM, RUSHMORE, VENETIANbest match: STARTING WITH ROCK BANDS (3/4)","STARTING WITH ROCK BANDSCREAMSICLE, JOURNEYMAN, KISSCAM, RUSHMORE"


Predicted,Actual
"✓ LEADERSHIP TITLESCHAIR, CHIEF, DIRECTOR, HEAD","PERSON IN CHARGECHAIR, CHIEF, DIRECTOR, HEAD"
"✓ NATURE TERMSFIELD, GREEN, GROUNDS, LAWN","GRASSY AREAFIELD, GREEN, GROUNDS, LAWN"
"✗ COLOR TERMSBROWN, HAZEL, PEA, COCObest match: WORDS BEFORE “NUT” (3/4)","WORDS BEFORE “NUT”CHEST, COCO, HAZEL, PEA"
"✗ FICTITIOUS/UNCHAINED CONCEPTSFICTION, DOGS, UNCHAINED, CHAIRbest match: SECOND WORDS IN TARANTINO MOVIES (3/4)","SECOND WORDS IN TARANTINO MOVIESBROWN, DOGS, FICTION, UNCHAINED"


Contest,Solved,Group Acc,Partial-3,Partial-2
"NYT Connections 172 - November 30th, 2023",✗,50%,75%,100%
"NYT Connections 314 - April 20th, 2024",✗,25%,75%,100%
"NYT Connections 283 - March 20th, 2024",✗,50%,100%,100%
"NYT Connections 304 - April 10th, 2024",✗,50%,100%,100%
Average (4),0.0%,43.8%,87.5%,100.0%


In [40]:
#Baseline: no hints provied to GPT
raw_replies_without_hints = []
all_actual_without_hints = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. 

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}

        Think step by step and use your knowledge to find the groups. 
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]
        """}
        ]  

    # Test GPT with Baseline (no hints)
    print("\nTesting GPT with Baseline (no hints) grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    
    raw_reply = ask_gpt(messages, group_prompt)
    raw_replies_without_hints.append(raw_reply)
    all_actual_without_hints.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")



Testing GPT with Baseline (no hints) grouping...
Tools: None
  [1/4] done: NYT Connections 172 - November 30th, 2023

Testing GPT with Baseline (no hints) grouping...
Tools: None
  [2/4] done: NYT Connections 314 - April 20th, 2024

Testing GPT with Baseline (no hints) grouping...
Tools: None
  [3/4] done: NYT Connections 283 - March 20th, 2024

Testing GPT with Baseline (no hints) grouping...
Tools: None
  [4/4] done: NYT Connections 304 - April 10th, 2024


In [41]:
scores_no_hints, summary_no_hints = evaluate_run(
    raw_replies_without_hints,
    all_actual_without_hints,
)

Predicted,Actual
"✓ WORDS THAT MEAN TO AVOIDDODGE, DUCK, ESCAPE, SKIRT","AVOIDDODGE, DUCK, ESCAPE, SKIRT"
"✗ BIRDSBIRDS, GEESE, ROBIN, HOBBESbest match: SIDEKICKS (2/4)","HITCHCOCK MOVIESBIRDS, NOTORIOUS, REBECCA, ROPE"
"✗ CHARACTERS FROM LITERATUREREBECCA, WATSON, NOTORIOUS, HOBBESbest match: HITCHCOCK MOVIES (2/4)","SIDEKICKSGOOSE, HOBBES, ROBIN, WATSON"
"✓ COMMON PHRASESCREAM, STRING, SAY, COTTAGE","___ CHEESECOTTAGE, CREAM, SAY, STRING"


Predicted,Actual
"✓ NONSENSE WORDSBUNK, CROCK, HOGWASH, HORSEFEATHERS","BALDERDASHBUNK, CROCK, HOGWASH, HORSEFEATHERS"
"✗ BINDING KNOTSBOWLINE, HITCH, SHEEPSHANK, BATONbest match: TYPES OF KNOTS (3/4)","TRACK AND FIELD EQUIPMENTBATON, HAMMER, HURDLE, POLE"
"✗ FARM TOOLSHAMMER, HURDLE, POLE, PITCHFORKbest match: TRACK AND FIELD EQUIPMENT (3/4)","PARTS OF A DEVIL COSTUMEGOATEE, HORNS, PITCHFORK, TAIL"
"✗ ANIMAL PARTSGOATEE, HORNS, TAIL, BENDbest match: PARTS OF A DEVIL COSTUME (3/4)","TYPES OF KNOTSBEND, BOWLINE, HITCH, SHEEPSHANK"


Predicted,Actual
"✓ SPORTS TERMSASTROTURF, JUMBOTRON, SCOREBOARD, SKYBOX","SEEN AT A SPORTS STADIUMASTROTURF, JUMBOTRON, SCOREBOARD, SKYBOX"
"✓ CAMERA BRANDSFUJIFILM, HASSELBLAD, OLYMPUS, POLAROID","CAMERA BRANDSFUJIFILM, HASSELBLAD, OLYMPUS, POLAROID"
"✓ PASTA DISHESBOLOGNESE, NEAPOLITAN, PARMESAN, VENETIAN","ITALIAN DEMONYMSBOLOGNESE, NEAPOLITAN, PARMESAN, VENETIAN"
"✓ ICE CREAM FLAVORSCREAMSICLE, JOURNEYMAN, KISSCAM, RUSHMORE","STARTING WITH ROCK BANDSCREAMSICLE, JOURNEYMAN, KISSCAM, RUSHMORE"


Predicted,Actual
"✗ COLORSGREEN, BROWN, HAZEL, COCObest match: WORDS BEFORE “NUT” (2/4)","PERSON IN CHARGECHAIR, CHIEF, DIRECTOR, HEAD"
"✓ LEADERSHIP TITLESCHAIR, CHIEF, DIRECTOR, HEAD","GRASSY AREAFIELD, GREEN, GROUNDS, LAWN"
"✗ TYPES OF GROUNDSFIELD, LAWN, GROUNDS, DOGSbest match: GRASSY AREA (3/4)","WORDS BEFORE “NUT”CHEST, COCO, HAZEL, PEA"
"✗ GENRES OF LITERATUREFICTION, UNCHAINED, PEA, CHESTbest match: WORDS BEFORE “NUT” (2/4)","SECOND WORDS IN TARANTINO MOVIESBROWN, DOGS, FICTION, UNCHAINED"


Contest,Solved,Group Acc,Partial-3,Partial-2
"NYT Connections 172 - November 30th, 2023",✗,50%,50%,100%
"NYT Connections 314 - April 20th, 2024",✗,25%,100%,100%
"NYT Connections 283 - March 20th, 2024",✓,100%,100%,100%
"NYT Connections 304 - April 10th, 2024",✗,25%,50%,100%
Average (4),25.0%,50.0%,75.0%,100.0%


In [43]:
## Comparison: No Hints vs With Hints
from IPython.display import display, HTML

metrics = ["gcr", "acc", "p3", "p2"]
labels  = ["Game Completion Rate", "Group Accuracy", "Partial-3 Rate", "Partial-2 Rate"]

def _delta_cell(a, b):
    diff = b - a
    color = "#66bb6a" if diff > 0 else ("#ef5350" if diff < 0 else "#aaa")
    arrow = "▲" if diff > 0 else ("▼" if diff < 0 else "—")
    return (f'<td style="padding:6px 14px;text-align:center;color:{color};font-weight:bold">'
            f'{arrow} {abs(diff):.1%}</td>')

header = (
    '<tr style="color:#aaa;border-bottom:1px solid #555">'
    '<th style="padding:6px 14px;text-align:left">Metric</th>'
    '<th style="padding:6px 14px">No Hints (A)</th>'
    '<th style="padding:6px 14px">With Hints (B)</th>'
    '<th style="padding:6px 14px">Δ (B − A)</th>'
    '</tr>'
)
rows = "".join(
    f'<tr>'
    f'<td style="padding:6px 14px">{lbl}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_no_hints[m]:.1%}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_rag[m]:.1%}</td>'
    f'{_delta_cell(summary_no_hints[m], summary_rag[m])}'
    f'</tr>'
    for m, lbl in zip(metrics, labels)
)

display(HTML(
    '<h2 style="font-family:monospace;border-bottom:2px solid #555;padding-bottom:6px">'
    'Comparison: No Hints vs With WordNet Hints</h2>'
    '<table style="border-collapse:collapse;font-family:monospace;font-size:0.95em">'
    f'{header}{rows}</table>'
    '<p style="color:#888;font-size:0.85em;font-family:monospace">'
    '▲ = With-hints better &nbsp;|&nbsp; ▼ = No-hints better</p>'
))

Metric,No Hints (A),With Hints (B),Δ (B − A)
Game Completion Rate,25.0%,0.0%,▼ 25.0%
Group Accuracy,50.0%,43.8%,▼ 6.2%
Partial-3 Rate,75.0%,87.5%,▲ 12.5%
Partial-2 Rate,100.0%,100.0%,— 0.0%
